In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

subscription=os.getenv("SUBSCRIPTION")
resource_group=os.getenv("RESOURCE_GROUP")
ws_name=os.getenv("WS_NAME")
print(f"Subscription: {subscription}")
print(f"Resource Group: {resource_group}")  
print(f"Workspace Name: {ws_name}")

Subscription: 784e6c68-d702-4a8b-a678-2e0f2f36deab
Resource Group: az-ml-chris
Workspace Name: AZ-ML-Chris


In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

# authenticate
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

SUBSCRIPTION = os.getenv("SUBSCRIPTION")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
WS_NAME = os.getenv("WS_NAME")
# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WS_NAME,
)

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Update the 'my_path' variable to match the location of where you downloaded the data on your
# local filesystem

my_path = "./data/default_of_credit_card_clients.csv"
# Set the version number of the data asset
v1 = "initial"

my_data = Data(
    name="credit-card",
    version=v1,
    description="Credit card data",
    path=my_path,
    type=AssetTypes.URI_FILE,
)

## Create data asset if it doesn't already exist:
try:
    data_asset = ml_client.data.get(name="credit-card", version=v1)
    print(
        f"Data asset already exists. Name: {my_data.name}, version: {my_data.version}"
    )
except:
    ml_client.data.create_or_update(my_data)
    print(f"Data asset created. Name: {my_data.name}, version: {my_data.version}")

In [5]:
import pandas as pd

# Get a handle of the data asset and print the URI
data_asset = ml_client.data.get(name="credit-card", version="initial")
print(f"Data asset URI: {data_asset.path}")

Data asset URI: azureml://subscriptions/784e6c68-d702-4a8b-a678-2e0f2f36deab/resourcegroups/az-ml-chris/workspaces/AZ-ML-Chris/datastores/workspaceblobstore/paths/LocalUpload/e23fa14f1dba6d0c9b5e51ea5772df54fa59a1eb80f7b20cff8efb56da4accb2/default_of_credit_card_clients.csv


In [ ]:
from azureml.fsspec import AzureMachineLearningFileSystem

fs = AzureMachineLearningFileSystem(data_asset.path)